In [11]:
pip install statsmodels

  Using cached statsmodels-0.14.6-cp312-cp312-win_amd64.whl.metadata (9.8 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached statsmodels-0.14.6-cp312-cp312-win_amd64.whl (9.5 MB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)

   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   --

In [12]:
# ============================================
# 1. IMPORTS AND SETUP
# ============================================

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from sklearn.decomposition import PCA
from pathlib import Path

RESULTS_DIR = Path("../results")

In [13]:
# ============================================
# 2. LOAD AND MERGE DATA
# ============================================

meta = pd.read_csv(RESULTS_DIR / "analytical_dataset.csv")
mds = pd.read_csv(RESULTS_DIR / "mds_scores.csv")
df = meta.merge(mds, on="id_hash", how="inner")
print("Merged N:", len(df))

Merged N: 5136


In [14]:
# ============================================
# 3. RQ1: MDS vs Engagement Metrics
#    (Spearman correlations)
# ============================================

print("\n--- RQ1 ---")
for col in ["log_plays","log_likes","log_comments","log_shares"]:
    r, p = stats.spearmanr(df["MDS"], df[col])
    print(f"  MDS vs {col}: rho={r:+.4f} p={p:.4g}")


--- RQ1 ---
  MDS vs log_plays: rho=+0.0360 p=0.009959
  MDS vs log_likes: rho=+0.0370 p=0.008038
  MDS vs log_comments: rho=+0.0138 p=0.3212
  MDS vs log_shares: rho=-0.0154 p=0.2691


In [15]:
# ============================================
# 4. RQ2: Modality-Pair Similarities vs Engagement
#    (Which modality pair matters most?)
# ============================================

print("\n--- RQ2 ---")
for pair in ["sim_text_audio","sim_text_video","sim_audio_video"]:
    r, p = stats.spearmanr(df[pair], df["log_plays"])
    print(f"  {pair} vs log_plays: rho={r:+.4f} p={p:.4g}")


--- RQ2 ---
  sim_text_audio vs log_plays: rho=-0.0358 p=0.01023
  sim_text_video vs log_plays: rho=-0.0163 p=0.2443
  sim_audio_video vs log_plays: rho=-0.0159 p=0.2552


In [16]:
# ============================================
# 5. RQ3: Controlled Regression Analysis
#    (Controlling for followers, ad, duet, category)
# ============================================

print("\n--- RQ3 ---")
d2 = pd.get_dummies(df.copy(), columns=["category"], drop_first=True)
cat_cols = [c for c in d2.columns if c.startswith("category_")]
d2["is_ad_bin"] = (d2["is_ad"].astype(str) == "yes").astype(int)
d2["is_duet_bin"] = (d2["is_duet"].astype(str) == "yes").astype(int)

X = sm.add_constant(d2[["MDS","log_followers","is_ad_bin","is_duet_bin"] + cat_cols].astype(float))
y = d2["log_plays"].astype(float)
model = sm.OLS(y, X).fit()
print(model.summary().tables[1])


--- RQ3 ---
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     12.5937      0.103    122.470      0.000      12.392      12.795
MDS                        0.0034      0.054      0.062      0.951      -0.103       0.109
log_followers              0.0894      0.008     11.640      0.000       0.074       0.104
is_ad_bin                  0.2686      0.173      1.553      0.120      -0.070       0.608
is_duet_bin               -0.4135      0.312     -1.325      0.185      -1.025       0.198
category_comedy_drame      1.5235      0.077     19.721      0.000       1.372       1.675
category_dance_music       1.0458      0.077     13.573      0.000       0.895       1.197
category_entertaiment      2.4525      0.077     31.846      0.000       2.302       2.603
category_fashion           1.0134      0.077     13.095      0.000       0.86

In [17]:
# ============================================
# 6. RQ4: Trained vs Untrained MDS Comparison
#    (Compare alignment quality before/after training)
# ============================================

print("\n--- RQ4 ---")
text = np.load(RESULTS_DIR / "text_embeddings_aligned.npy")
audio = np.load(RESULTS_DIR / "audio_embeddings_aligned.npy")
video = np.load(RESULTS_DIR / "video_embeddings_aligned.npy")

def to128(mat):
    mat = (mat - mat.mean(0)) / (mat.std(0) + 1e-6)
    return PCA(n_components=128, random_state=42).fit_transform(mat)

t128, a128, v128 = to128(text), to128(audio), to128(video)

def cos(a, b):
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return (a * b).sum(1)

raw_mds = 1 - (cos(t128, v128) + cos(t128, a128)) / 2
df["raw_MDS"] = raw_mds


--- RQ4 ---


In [18]:
# ============================================
# 7. RQ4: Correlation Comparison
#    (Compare trained vs untrained correlations)
# ============================================

r_trained, p_trained = stats.spearmanr(df["MDS"], df["log_plays"])
r_untrained, p_untrained = stats.spearmanr(df["raw_MDS"], df["log_plays"])
r_agree, p_agree = stats.spearmanr(df["MDS"], df["raw_MDS"])

print(f"  Trained MDS vs log_plays:   rho={r_trained:+.4f} p={p_trained:.4g}")
print(f"  Untrained MDS vs log_plays: rho={r_untrained:+.4f} p={p_untrained:.4g}")
print(f"  Trained vs untrained agreement: rho={r_agree:+.4f} p={p_agree:.4g}")

  Trained MDS vs log_plays:   rho=+0.0360 p=0.009959
  Untrained MDS vs log_plays: rho=+0.0228 p=0.1021
  Trained vs untrained agreement: rho=+0.0266 p=0.05653


In [19]:
# ============================================
# 8. SAVE FINAL ANALYTICAL DATASET
# ============================================

df.to_csv(RESULTS_DIR / "analytical_dataset_final.csv", index=False)
print("\nSaved analytical_dataset_final.csv")
print(f"Final dataset shape: {df.shape}")


Saved analytical_dataset_final.csv
Final dataset shape: (5136, 31)


In [20]:
# ============================================
# 10. VISUALIZATION HELPER
#     (Optional - uncomment to use)
# ============================================

"""
import matplotlib.pyplot as plt
import seaborn as sns

# Scatter plot: MDS vs log_plays
plt.figure(figsize=(10, 6))
plt.scatter(df['MDS'], df['log_plays'], alpha=0.3, s=10)
plt.xlabel('MDS (Modality Distance Score)')
plt.ylabel('Log Plays')
plt.title('MDS vs Engagement')
plt.grid(True, alpha=0.3)
plt.show()

# Box plot: MDS by category
plt.figure(figsize=(12, 6))
df.boxplot(column='MDS', by='category', rot=45)
plt.title('MDS Distribution by Category')
plt.suptitle('')
plt.tight_layout()
plt.show()
"""

"\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\n# Scatter plot: MDS vs log_plays\nplt.figure(figsize=(10, 6))\nplt.scatter(df['MDS'], df['log_plays'], alpha=0.3, s=10)\nplt.xlabel('MDS (Modality Distance Score)')\nplt.ylabel('Log Plays')\nplt.title('MDS vs Engagement')\nplt.grid(True, alpha=0.3)\nplt.show()\n\n# Box plot: MDS by category\nplt.figure(figsize=(12, 6))\ndf.boxplot(column='MDS', by='category', rot=45)\nplt.title('MDS Distribution by Category')\nplt.suptitle('')\nplt.tight_layout()\nplt.show()\n"